# Download Data from FTP Server

if you want to download the massive data, you can go to the [Zenodo](https://zenodo.org/records/14967861) to download.

In [2]:
import ftplib
from dataclasses import dataclass
from pathlib import Path
from tqdm import tqdm

@dataclass
class Item:
    name: str
    is_dir: bool
    size: int | None

def get_directory_contents(ftp: ftplib.FTP, remote_dir: str) -> dict[str, Item]:
    items: dict[str, Item] = {}
    try:
        ftp.cwd(remote_dir)
        lines: list[str] = []
        ftp.retrlines('LIST', lines.append)
        for line in lines:
            parts = line.split(None, 8)
            if len(parts) < 9:
                continue

            name = parts[-1]
            if name in ['.', '..']:
                continue
            
            is_dir = line.startswith('d') or line.startswith('l')
            size = int(parts[4]) if not is_dir and parts[4].isdigit() else None
            items[name] = Item(name, is_dir, size)
    
    except Exception as e:
        print(f"ERROR: Accessing {remote_dir} failed: {e}")
    
    return items

def process_item(ftp: ftplib.FTP, item: Item, current_remote_dir: str, local_parent_dir: Path):
    next_remote_dir = f"{current_remote_dir}/{item.name}".replace('//', '/')
    local_save_path = local_parent_dir / item.name
    
    if item.is_dir:
        local_save_path.mkdir(exist_ok=True, parents=True)
        sub_contents = get_directory_contents(ftp, next_remote_dir)
        for sub_item in sub_contents.values():
            process_item(ftp, sub_item, next_remote_dir, local_save_path)
        
        ftp.cwd(current_remote_dir)
    else:
        local_save_path.parent.mkdir(exist_ok=True, parents=True)
        ftp.cwd(current_remote_dir)
        
        # Skip if file already exists with same size
        if local_save_path.exists() and local_save_path.stat().st_size == item.size:
            print(f"SKIP: {item.name} already exists.")
            return

        with open(local_save_path, 'wb') as f:
            with tqdm(total=item.size, unit='B', unit_scale=True, desc=item.name, leave=True) as pbar:
                def callback(data):
                    f.write(data)
                    pbar.update(len(data))
                
                ftp.retrbinary(f"RETR {item.name}", callback, blocksize=32768)

ftp_host = "massive-ftp.ucsd.edu"
train_data_path = "/v01/MSV000081382/peak/DeepNovo/HighResolution/data/cross.9high_80k.exclude_mouse/cross.cat.mgf.train.repeat"
valid_data_path = "/v01/MSV000081382/peak/DeepNovo/HighResolution/data/cross.9high_80k.exclude_mouse/cross.cat.mgf.valid.repeat"
test_data_path = "/v01/MSV000081382/peak/DeepNovo/HighResolution/data/high.mouse.PXD004948/peaks.db.mgf"

print(f"Connecting to {ftp_host}...")
ftp = ftplib.FTP(ftp_host, timeout=60)
ftp.login()
print("Login success.")

for remote_path in [test_data_path, valid_data_path, train_data_path]:
    ftp.cwd(str(Path(remote_path).parent))
    contents = get_directory_contents(ftp, str(Path(remote_path).parent))
    # repalce with your own local output directory
    base_output_dir = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/cross.9high_80k.exclude_mouse")
    if Path(remote_path).name not in contents:
        print(f"WARNING: Target '{Path(remote_path).name}' not found.")
        continue
    
    print(f"Processing: {Path(remote_path).name}")
    process_item(ftp, contents[Path(remote_path).name], str(Path(remote_path).parent), base_output_dir)

ftp.quit()
print("All downloads finished.")

Connecting to massive-ftp.ucsd.edu...
Login success.
Processing: peaks.db.mgf


peaks.db.mgf: 100%|██████████| 60.1M/60.1M [00:38<00:00, 1.55MB/s]  


Processing: cross.cat.mgf.valid.repeat


cross.cat.mgf.valid.repeat: 100%|██████████| 97.8M/97.8M [00:49<00:00, 1.98MB/s]   


Processing: cross.cat.mgf.train.repeat


cross.cat.mgf.train.repeat: 100%|██████████| 1.68G/1.68G [08:35<00:00, 3.26MB/s]   


All downloads finished.


# Count all residue types in the dataset and map them to the MassIVE KB format.

May be you should view all the data to check the presence of PTM

In [4]:
import re
from pathlib import Path
from collections import defaultdict

import pandas as pd
from tqdm import tqdm
from pyteomics.mgf import MGF

def count_residues(data_path: Path):
    counter = defaultdict(int)
    with MGF(str(data_path)) as spectra:
        for spectrum in tqdm(spectra, desc=str(data_path.stem), unit="spectra"):
            seq = spectrum["params"].get("seq")
            residues = re.split(r"(?<=.)(?=[A-Z])", seq)
            for res in residues:
                counter[res] += 1

    residues = [(res, counter[res]) for res in counter.keys()]
    return pd.DataFrame(residues, columns=["Residue", "Count"])

dir_path = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data")
data_path = dir_path.joinpath("cross.9high_80k.exclude_mouse").joinpath("peaks.db.mgf")
count_residues(data_path)

peaks.db: 37021spectra [00:04, 7763.59spectra/s]


,Residue,Count
0,I,54632
1,A,40839
2,H,5770
3,Y,13813
4,N,20706
5,K,21794
6,R,17906
7,V,32965
8,E,48956
9,D,34477


# Write to a new MGF file with the transformed PTMs.

The format you can check the `rocnovo/tokenizer/peptide.py`

In [2]:
import re
from pathlib import Path

from tqdm import tqdm
from pyteomics.mgf import MGF, write

mapping = {
    "C(+57.02)": "C+57.021",
    "N(+.98)": "N+0.984",
    "M(+15.99)": "M+15.995",
    "Q(+.98)": "Q+0.984"
}

def process_spectra_stream(path: Path):
    with MGF(str(path)) as spectra:
        for spectrum in tqdm(spectra, desc=str(path.stem), unit="spectra"):
            seq = spectrum["params"].get("seq")
            residues = re.split(r"(?<=.)(?=[A-Z])", seq)
            seq = "".join([mapping.get(res, res) for res in residues])
            spectrum["params"]["seq"] = seq
            yield spectrum

dir_path = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data")

data_path = dir_path.joinpath("cross.9high_80k.exclude_mouse").joinpath("peaks.db.mgf")
output_path = dir_path.joinpath("processed").joinpath("peaks.db.mgf")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as out_file:
    write(process_spectra_stream(data_path), out_file)

peaks.db: 0spectra [00:00, ?spectra/s]

peaks.db: 37021spectra [00:45, 820.30spectra/s] 


# Converting Parquet Data

Currently, we only support the MGF format. Support for other formats, such as Parquet, mzXML, and mzML, will be gradually introduced in the future.

MGF, mzXML, and mzML files all need to be converted into the HDF5 format.

For Parquet data, we will subsequently develop custom Dataset and DataLoader classes specifically for data loading.

# Download Novobench Data

In [1]:
import requests
from pathlib import Path

from tqdm import tqdm

novobench_base_url = "https://huggingface.co/datasets/jingbo02/NovoBench/resolve/main/data"
subsets = ["seven_species", "nine_species", "hc_pt"]
datasets = ["train.parquet", "valid.parquet", "test.parquet"]

url = "https://huggingface.co/datasets/jingbo02/NovoBench/resolve/main/data"
local_dir = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/parquet")
local_dir.mkdir(parents=True, exist_ok=True)

for subset in subsets:
    for dataset in datasets:
        local_filename = local_dir / subset / dataset
        local_filename.parent.mkdir(parents=True, exist_ok=True)
        url = f"{novobench_base_url}/{subset}/{dataset}"
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(local_filename, 'wb') as f: 
                with tqdm(desc=f"{subset}/{dataset}", total=int(r.headers['Content-Length']), unit='B', unit_scale=True) as pbar:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))

print("Download Success.")

seven_species/train.parquet: 100%|██████████| 1.41G/1.41G [01:00<00:00, 23.5MB/s]  
seven_species/valid.parquet: 100%|██████████| 86.8M/86.8M [00:04<00:00, 21.6MB/s]
seven_species/test.parquet: 100%|██████████| 81.8M/81.8M [00:04<00:00, 19.9MB/s]
nine_species/train.parquet: 100%|██████████| 713M/713M [00:30<00:00, 23.4MB/s]   
nine_species/valid.parquet: 100%|██████████| 42.3M/42.3M [00:02<00:00, 18.6MB/s]
nine_species/test.parquet: 100%|██████████| 38.7M/38.7M [00:02<00:00, 18.4MB/s]
hc_pt/train.parquet: 100%|██████████| 312M/312M [00:06<00:00, 46.1MB/s] 
hc_pt/valid.parquet: 100%|██████████| 38.4M/38.4M [00:02<00:00, 18.0MB/s]
hc_pt/test.parquet: 100%|██████████| 39.4M/39.4M [00:01<00:00, 20.9MB/s]

Download Success.


In [32]:
import re
from pathlib import Path
from collections import defaultdict

import pandas as pd

def count_residues(dir_path: Path):
    counter = defaultdict(int)
    for file_path in sorted(dir_path.iterdir()):
        if file_path.is_dir():
            subset_counter = count_residues(file_path)

            for res, count in subset_counter.items():
                counter[res] += count
        
        elif file_path.suffix == ".parquet":
            df = pd.read_parquet(file_path, engine="pyarrow")
            for _, row in tqdm(df.iterrows(), desc=str(f"{file_path.parent.stem}/{file_path.stem}"), total=len(df)):
                seq = row["modified_sequence"]
                residues = re.split(r"(?<=.)(?=[A-Z])", seq)
                for res in residues:
                    counter[res] += 1
    
    return counter

def get_counter_df(counter: dict):
    residues = [(res, counter[res]) for res in counter.keys()]
    df = pd.DataFrame(residues, columns=["Residue", "Count"])
    df.sort_values(by="Residue").reset_index(drop=True, inplace=True)
    return df

nine_species_dir_path = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/parquet/nine_species")
seven_species_dir_path = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/parquet/seven_species")
hc_pt_dir_path = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/parquet/hc_pt")
nine_species_counter = count_residues(nine_species_dir_path)
seven_species_counter = count_residues(seven_species_dir_path)
hc_pt_counter = count_residues(hc_pt_dir_path)
nine_species_df = get_counter_df(nine_species_counter)
seven_species_df = get_counter_df(seven_species_counter)
hc_pt_df = get_counter_df(hc_pt_counter)

hc_pt/valid: 100%|██████████| 25718/25718 [00:01<00:00, 20955.89it/s]


In [29]:
nine_species_df

,Residue,Count
0,T,474329
1,P,449340
2,G,664537
3,R,291552
4,E,665723
5,D,544933
6,A,691034
7,K,405401
8,V,630646
9,Q,323821


In [30]:
seven_species_df

,Residue,Count
0,I,836100
1,D,352299
2,A,519631
3,S,338025
4,H,119983
5,R,193241
6,G,439073
7,V,436531
8,Y,149484
9,P,286112


In [31]:
hc_pt_df

,Residue,Count
0,A,255740
1,Y,100282
2,Q,136570
3,V,206744
4,L,368679
5,P,182802
6,K,201585
7,E,253653
8,T,173783
9,F,150747


# Note: Fixed Modifications of Cysteine

Note that among the three datasets mentioned above, cysteine residues in the nine-species and seven-species datasets are treated with fixed modifications, whereas the HC PT dataset is not.

To facilitate unified data processing, we also mapped the cysteines in the HC PT dataset to fixed modifications. This will not affect model training or the final evaluation results.
(This deterministic one-to-one mapping does not impact model training or final evaluation, as both the mass dictionary and ground truth labels are updated synchronously.)

In [39]:
import numpy as np
from pyteomics.mgf import write

mapping = {
    "C": "C+57.021",
    "C(+57.02)": "C+57.021",
    "N(+.98)": "N+0.984",
    "M(ox)": "M+15.995",
    "M(+15.99)": "M+15.995",
    "Q(+.98)": "Q+0.984"
}

def process_spectra_stream(path: Path):
    df = pd.read_parquet(path, engine="pyarrow")
    for i, row in tqdm(df.iterrows(), desc=str(f"{path.parent.stem}/{path.stem}"), total=len(df)):
        seq = row["modified_sequence"]
        residues = re.split(r"(?<=.)(?=[A-Z])", seq)
        seq = "".join([mapping.get(res, res) for res in residues])
        mz_array = np.array(row.mz_array, dtype=np.float64)
        int_array = np.array(row.intensity_array, dtype=np.float32)
        spectrum = {
            "params": {
                "title": f"{path.parent.stem}:{path.stem}:{i}",
                "pepmass": row["precursor_mz"],
                "charge": int(row["precursor_charge"]),
                "seq": seq,
                "ms_level": 2,
            },
            "m/z array": mz_array,
            "intensity array": int_array
        }
        yield spectrum

dir_path = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/parquet")
output_dir = Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed")
for sub_dir in sorted(dir_path.iterdir()):
    sub_output_dir = output_dir.joinpath(sub_dir.stem)
    sub_output_dir.mkdir(exist_ok=True, parents=True)
    
    for file_path in sorted(sub_dir.iterdir()):
        output_path = sub_output_dir.joinpath(f"{file_path.stem}.mgf")
        with open(output_path, "w") as f:
            write(process_spectra_stream(file_path), f)

seven_species/valid: 100%|██████████| 17740/17740 [00:16<00:00, 1096.30it/s]


# Convert MGF data to hdf5 format

In [2]:
import sys
sys.path.append("..")
from pathlib import Path

from rocnovo.data.io import MgfToHdf5

def convert_mgf2hdf5(path: Path):
    if path.is_file() and path.suffix == ".mgf":
        MgfToHdf5(
            path,
            path.with_suffix(".hdf5"),
            annotated=True
        )
    
    elif path.is_dir():
        for p in sorted(path.iterdir()):
            convert_mgf2hdf5(p)

convert_mgf2hdf5(Path("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed"))

/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/hc_pt/test.mgf: 26536spectra [00:07, 3727.90spectra/s]
/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/hc_pt/train.mgf: 213284spectra [01:00, 3506.70spectra/s]
/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/hc_pt/valid.mgf: 25718spectra [00:06, 3917.87spectra/s]
/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/nine_species/test.mgf: 27142spectra [00:04, 5486.81spectra/s]
/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/nine_species/train.mgf: 499402spectra [01:29, 5554.97spectra/s] 
/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/nine_species/valid.mgf: 28572spectra [00:05, 5346.29spectra/s]
/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/peaks.db.mgf: 37021spectra [00:04, 8857.13spectra/s]
/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/seven_species/test.mgf: 17094spectra [00:09, 1868.21spectra/s]
/data2/xp/RocNovo-Lightning/outputs/tu